# BQ-KFE: Quantum-Assisted Dual-Channel Healthcare IIoT Security
**End-to-end executable research prototype — Edge-IIoTset**

This notebook implements the eight-stage pipeline: dataset acquisition, temporal feature construction, dual-channel Q-DCTD detection, optional variational-quantum calibration, post-quantum cryptography where `liboqs-python` is available, a hash-chained permissioned-ledger prototype, mitigation orchestration, and evaluation/ablation.

> **Scope and reproducibility notes**
> - The supplied KaggleHub dataset is the primary benchmark. Dataset columns are discovered at runtime because releases/files can differ.
> - Edge-IIoTset is primarily network/security traffic data. If genuine CPU, memory, or sensor telemetry columns are absent, the notebook explicitly creates a **network-behavior proxy channel**; it never labels that proxy as measured device telemetry.
> - A local hash-chained ledger is provided as an executable blockchain prototype. It is not a deployed Hyperledger Fabric network. Fabric deployment requires a separately configured Fabric network, identities, peers, ordering service, and chaincode deployment.
> - ML-KEM and ML-DSA are used only when the Open Quantum Safe (`liboqs`) bindings expose them. Otherwise, the notebook uses clearly labelled classical X25519/Ed25519 fallback for functional pipeline testing. The fallback is **not post-quantum**.
> - PennyLane is optional. If installed, the notebook runs a small variational circuit for offline calibration; otherwise, a classical calibration fallback is used and reported as such.
> - Results are produced from the downloaded dataset; no benchmark scores are hard-coded.


## 0. Environment and reproducibility

In [ ]:
# Run once in a fresh environment if packages are missing.
%pip -q install kagglehub pandas numpy scikit-learn matplotlib seaborn tqdm joblib torch cryptography openpyxl
# Optional quantum and post-quantum packages:
# %pip -q install pennylane
# liboqs-python may require a separately installed liboqs native library:
# %pip -q install liboqs-python


In [ ]:
import os, re, json, time, math, hashlib, warnings, random, platform, subprocess, sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.impute import SimpleImputer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
print("Python:", platform.python_version(), "| Torch device:", DEVICE)


## Stage 1 — Dataset acquisition and threat definition

In [ ]:
import kagglehub

path = kagglehub.dataset_download("mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot")
DATA_ROOT = Path(path)
print("Path to dataset files:", DATA_ROOT)

all_files = [p for p in DATA_ROOT.rglob("*") if p.is_file()]
print("Files discovered:", len(all_files))
for p in all_files[:40]:
    print(p.relative_to(DATA_ROOT), f"({p.stat().st_size/1e6:.1f} MB)")


In [ ]:
# Load CSV files. Large files are read normally; set ROW_LIMIT for a quick smoke test.
ROW_LIMIT = None  # e.g. 250_000 for a quick test; None loads complete CSVs
csv_files = [p for p in all_files if p.suffix.lower() in {".csv", ".txt"}]
if not csv_files:
    raise FileNotFoundError("No CSV/TXT files found under the KaggleHub download directory.")

frames = []
for fp in csv_files:
    try:
        df_part = pd.read_csv(fp, low_memory=False, nrows=ROW_LIMIT)
        df_part["_source_file"] = fp.name
        frames.append(df_part)
        print(f"Loaded {fp.name}: {df_part.shape}")
    except Exception as e:
        print("Skipped", fp.name, "->", repr(e))

if not frames:
    raise RuntimeError("No dataset file could be read as CSV.")
raw = pd.concat(frames, ignore_index=True, sort=False)
print("Combined shape:", raw.shape)
display(raw.head())


In [ ]:
def norm_col(c):
    return re.sub(r"[^a-z0-9]+", "", str(c).strip().lower())

COL_LOOKUP = {norm_col(c): c for c in raw.columns}

def find_col(candidates):
    for cand in candidates:
        key = norm_col(cand)
        if key in COL_LOOKUP:
            return COL_LOOKUP[key]
    for c in raw.columns:
        k = norm_col(c)
        if any(norm_col(x) in k for x in candidates):
            return c
    return None

label_col = find_col(["Attack_type", "Attack", "Label", "class", "category"])
timestamp_col = find_col(["Timestamp", "Time", "Date"])
device_col = find_col(["device_id", "Device", "Device_Name", "src_ip", "Source IP", "IP"])
print("Detected label:", label_col, "| timestamp:", timestamp_col, "| device/group:", device_col)
if label_col is None:
    raise ValueError("Could not identify attack label column. Inspect raw.columns and set label_col manually.")
print(raw[label_col].astype(str).value_counts(dropna=False).head(30))


In [ ]:
# Normalize target labels into benign/malicious plus retained attack-family labels.
def clean_label(v):
    s = str(v).strip().lower()
    benign_tokens = {"benign", "normal", "0", "0.0", "normal traffic"}
    if s in benign_tokens or "benign" in s or s == "normal":
        return "Benign"
    if any(x in s for x in ["dos", "ddos", "denial"]): return "DoS/DDoS"
    if any(x in s for x in ["recon", "scan", "fingerprint"]): return "Reconnaissance"
    if any(x in s for x in ["inject", "sql", "xss", "command"]): return "Injection"
    if "malware" in s or "ransom" in s or "trojan" in s: return "Malware"
    if "mitm" in s or "man in the middle" in s or "man-in-the-middle" in s: return "Man-in-the-middle"
    if s in {"nan", "none", ""}: return "Unknown"
    return str(v).strip()

data = raw.copy()
data["attack_family"] = data[label_col].map(clean_label)
data = data[data["attack_family"] != "Unknown"].copy()
data["is_malicious"] = (data["attack_family"] != "Benign").astype(int)
print(data["attack_family"].value_counts())
print("Rows retained:", len(data))


## Stage 2 — Temporal alignment and dual-channel feature construction

In [ ]:
# Detect timestamp and device key. If no device identifier exists, use one global stream.
if timestamp_col is not None:
    data["_time"] = pd.to_datetime(data[timestamp_col], errors="coerce", utc=True)
else:
    data["_time"] = pd.NaT

if device_col is not None:
    data["_device"] = data[device_col].astype(str).fillna("unknown_device")
else:
    data["_device"] = "single_stream_no_device_id"

# Choose numeric features; exclude labels, identifiers, timestamps, and file marker.
exclude = {label_col, "attack_family", "is_malicious", "_source_file", "_time", "_device"}
if timestamp_col: exclude.add(timestamp_col)
if device_col: exclude.add(device_col)

numeric_cols = [c for c in data.select_dtypes(include=[np.number]).columns if c not in exclude]
# Convert numeric-looking object columns where a substantial fraction parses successfully.
for c in data.columns:
    if c in exclude or c in numeric_cols: continue
    parsed = pd.to_numeric(data[c], errors="coerce")
    if parsed.notna().mean() > 0.85:
        data[c] = parsed
        numeric_cols.append(c)

# Remove duplicate records and coerce invalid numeric values.
data = data.drop_duplicates().reset_index(drop=True)
for c in numeric_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

print("Numeric feature count:", len(numeric_cols))
print("Candidate numeric columns:", numeric_cols[:60])


In [ ]:
# Protocol/device categorical attributes: one-hot encode a bounded set of low-cardinality columns.
categorical_candidates = []
for c in data.columns:
    if c in exclude or c in numeric_cols: continue
    nunique = data[c].nunique(dropna=True)
    if 1 < nunique <= 30 and data[c].dtype == "object":
        categorical_candidates.append(c)
cat_cols = categorical_candidates[:12]
print("Categorical columns encoded:", cat_cols)

# One-hot encoding is applied before temporal windowing.
if cat_cols:
    cat_encoded = pd.get_dummies(data[cat_cols].fillna("missing").astype(str),
                                 prefix=cat_cols, dtype=np.float32)
else:
    cat_encoded = pd.DataFrame(index=data.index)

# Robust imputation/scaling for numeric features.
if not numeric_cols:
    raise ValueError("No numeric features detected. Inspect dataset and define numeric_cols manually.")
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()
X_num = imputer.fit_transform(data[numeric_cols])
X_num = scaler.fit_transform(X_num).astype(np.float32)
X_num_df = pd.DataFrame(X_num, columns=numeric_cols, index=data.index)
X_all_df = pd.concat([X_num_df, cat_encoded.set_index(data.index)], axis=1)

# Channel assignment. Device telemetry is only considered genuine when matching columns exist.
telemetry_patterns = ["cpu", "memory", "ram", "sensor", "temperature", "battery", "utilization"]
telemetry_cols = [c for c in X_all_df.columns if any(p in norm_col(c) for p in telemetry_patterns)]
network_patterns = ["packet", "flow", "duration", "proto", "byte", "port", "tcp", "udp",
                    "http", "dns", "icmp", "flag", "connection", "rate", "length"]
network_cols = [c for c in X_all_df.columns if any(p in norm_col(c) for p in network_patterns)]
if not network_cols:
    network_cols = list(X_all_df.columns)

telemetry_is_genuine = len(telemetry_cols) > 0
if telemetry_is_genuine:
    device_features = telemetry_cols
else:
    # Explicitly documented proxy channel from non-identical network behavioral features.
    device_features = [c for c in X_all_df.columns if c not in network_cols]
    if not device_features:
        device_features = list(X_all_df.columns)
print("Network channel:", len(network_cols), "features")
print("Device channel:", len(device_features), "features")
print("Device channel status:", "dataset telemetry columns found" if telemetry_is_genuine
      else "NETWORK-DERIVED PROXY (not measured device telemetry)")


In [ ]:
# Chronological/device-wise split first, then create windows within each partition.
# This prevents overlapping windows crossing train/validation/test boundaries.
WINDOW = 20
STRIDE = 5
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15

data = data.reset_index(drop=True)
X_all = X_all_df.to_numpy(dtype=np.float32)
y_text = data["attack_family"].astype(str).to_numpy()
devices = data["_device"].astype(str).to_numpy()
times = data["_time"]

def sort_group_indices(idx):
    if times.iloc[idx].notna().any():
        valid = times.iloc[idx].notna().to_numpy()
        valid_idx, invalid_idx = idx[valid], idx[~valid]
        valid_order = np.argsort(times.iloc[valid_idx].astype("int64").to_numpy())
        return np.concatenate([valid_idx[valid_order], invalid_idx])
    return idx

parts = {"train": [], "val": [], "test": []}
for dev in pd.unique(devices):
    idx = np.where(devices == dev)[0]
    idx = sort_group_indices(idx)
    n = len(idx)
    a, b = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
    parts["train"].extend(idx[:a])
    parts["val"].extend(idx[a:b])
    parts["test"].extend(idx[b:])

print({k: len(v) for k, v in parts.items()})

# Fit label encoder on all observed labels (labels are metadata, not model features).
label_encoder = LabelEncoder()
label_encoder.fit(y_text)
class_names = list(label_encoder.classes_)
y_encoded = label_encoder.transform(y_text).astype(np.int64)
print("Classes:", class_names)

def build_windows(indices):
    Xn, Xd, yy, groups = [], [], [], []
    indices = np.asarray(indices, dtype=int)
    # Re-segment independently per device and split.
    for dev in pd.unique(devices[indices]):
        ids = indices[devices[indices] == dev]
        ids = sort_group_indices(ids)
        if len(ids) < WINDOW: continue
        for start in range(0, len(ids) - WINDOW + 1, STRIDE):
            w = ids[start:start+WINDOW]
            Xn.append(X_all[w][:, [X_all_df.columns.get_loc(c) for c in network_cols]])
            Xd.append(X_all[w][:, [X_all_df.columns.get_loc(c) for c in device_features]])
            # Use final timestamp label for online/causal prediction.
            yy.append(y_encoded[w[-1]])
            groups.append(dev)
    if not Xn:
        return (np.empty((0, WINDOW, len(network_cols)), np.float32),
                np.empty((0, WINDOW, len(device_features)), np.float32),
                np.empty((0,), np.int64), np.array([]))
    return np.stack(Xn), np.stack(Xd), np.asarray(yy, np.int64), np.asarray(groups)

windows = {k: build_windows(v) for k, v in parts.items()}
for k, (xn, xd, yy, gg) in windows.items():
    print(k, "network", xn.shape, "device", xd.shape, "labels", yy.shape)

Xn_train, Xd_train, y_train, _ = windows["train"]
Xn_val, Xd_val, y_val, _ = windows["val"]
Xn_test, Xd_test, y_test, _ = windows["test"]
if min(len(y_train), len(y_val), len(y_test)) == 0:
    raise ValueError("At least one split has no complete temporal windows. Reduce WINDOW/STRIDE or inspect device groups.")


## Stage 3 — Proposed Q-DCTD dual-channel temporal detector

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class DualWindowDataset(Dataset):
    def __init__(self, xn, xd, y):
        self.xn = torch.tensor(xn, dtype=torch.float32)
        self.xd = torch.tensor(xd, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.xn[i], self.xd[i], self.y[i]

class ResidualTCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dilation, dropout=0.15):
        super().__init__()
        pad = (3 - 1) * dilation
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=pad, dilation=dilation)
        self.drop = nn.Dropout(dropout)
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.norm1 = nn.BatchNorm1d(out_ch); self.norm2 = nn.BatchNorm1d(out_ch)
    def _crop(self, x, length): return x[..., :length]
    def forward(self, x):
        length = x.shape[-1]
        z = self._crop(self.conv1(x), length)
        z = self.drop(torch.relu(self.norm1(z)))
        z = self._crop(self.conv2(z), length)
        z = self.drop(torch.relu(self.norm2(z)))
        return torch.relu(z + self.skip(x))

class TemporalEncoder(nn.Module):
    def __init__(self, in_features, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            ResidualTCNBlock(in_features, hidden, 1),
            ResidualTCNBlock(hidden, hidden, 2),
            ResidualTCNBlock(hidden, hidden, 4))
    def forward(self, x):
        z = self.net(x.transpose(1, 2))
        return z[:, :, -1]

class QDCTD(nn.Module):
    def __init__(self, n_network, n_device, n_classes, hidden=48):
        super().__init__()
        self.network_encoder = TemporalEncoder(n_network, hidden)
        self.device_encoder = TemporalEncoder(n_device, hidden)
        self.gate = nn.Sequential(nn.Linear(hidden*2, hidden), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(),
                                        nn.Dropout(0.2), nn.Linear(hidden, n_classes))
        self.binary_head = nn.Linear(hidden*2, 1)
    def forward(self, xn, xd):
        en = self.network_encoder(xn)
        ed = self.device_encoder(xd)
        g = self.gate(torch.cat([en, ed], dim=1))
        fused = torch.cat([g*en, (1-g)*ed], dim=1)
        return self.classifier(fused), self.binary_head(fused).squeeze(1), fused

BATCH_SIZE = 256
train_loader = DataLoader(DualWindowDataset(Xn_train, Xd_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(DualWindowDataset(Xn_val, Xd_val, y_val),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(DualWindowDataset(Xn_test, Xd_test, y_test),
                         batch_size=BATCH_SIZE, shuffle=False)

model = QDCTD(Xn_train.shape[-1], Xd_train.shape[-1], len(class_names)).to(DEVICE)
weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.ones(len(class_names), dtype=torch.float32)
for cls, w in zip(np.unique(y_train), weights): class_weights[int(cls)] = float(w)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(model)


In [ ]:
EPOCHS = 15  # Increase for final experiments after a smoke test.
history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
best_val = float("inf")
best_state = None

for epoch in range(EPOCHS):
    model.train(); train_losses = []
    for xn, xd, yy in train_loader:
        xn, xd, yy = xn.to(DEVICE), xd.to(DEVICE), yy.to(DEVICE)
        optimizer.zero_grad()
        logits, binary_logits, fused = model(xn, xd)
        loss = criterion(logits, yy)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval(); val_losses=[]; val_true=[]; val_pred=[]
    with torch.no_grad():
        for xn, xd, yy in val_loader:
            xn, xd, yy = xn.to(DEVICE), xd.to(DEVICE), yy.to(DEVICE)
            logits, _, _ = model(xn, xd)
            val_losses.append(criterion(logits, yy).item())
            val_true.extend(yy.cpu().numpy())
            val_pred.extend(logits.argmax(1).cpu().numpy())
    tr_loss = float(np.mean(train_losses)); va_loss = float(np.mean(val_losses))
    va_acc = accuracy_score(val_true, val_pred)
    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["val_accuracy"].append(va_acc)
    if va_loss < best_val:
        best_val = va_loss
        best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | train loss {tr_loss:.4f} | val loss {va_loss:.4f} | val acc {va_acc:.4f}")

if best_state is not None: model.load_state_dict(best_state)


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history["train_loss"], label="Training loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

def predict_loader(model, loader):
    model.eval(); probs=[]; ys=[]; embeddings=[]
    with torch.no_grad():
        for xn, xd, yy in loader:
            logits, _, fused = model(xn.to(DEVICE), xd.to(DEVICE))
            probs.append(torch.softmax(logits, 1).cpu().numpy())
            embeddings.append(fused.cpu().numpy())
            ys.append(yy.numpy())
    return np.concatenate(probs), np.concatenate(ys), np.concatenate(embeddings)

val_probs, val_y, val_embeddings = predict_loader(model, val_loader)
test_probs, test_y, test_embeddings = predict_loader(model, test_loader)


## Stage 4 — Offline variational quantum threat calibration

In [ ]:
# The circuit optimizes a compact set of calibration parameters on validation data.
# Quantum execution is optional; the notebook records which backend was actually used.
QUANTUM_BACKEND = "classical_fallback"
quantum_threshold = None
quantum_weights = None

try:
    import pennylane as qml
    from scipy.optimize import minimize

    n_qubits = 4
    dev_q = qml.device("default.qubit", wires=n_qubits, shots=None)

    @qml.qnode(dev_q, interface="autograd")
    def calibration_circuit(inputs, theta):
        for q in range(n_qubits):
            qml.RY(inputs[q], wires=q)
        for layer in range(2):
            for q in range(n_qubits):
                qml.RY(theta[layer, q], wires=q)
            for q in range(n_qubits-1):
                qml.CNOT(wires=[q, q+1])
        return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

    # Reduce validation features to four bounded angles.
    val_conf = val_probs.max(axis=1)
    val_correct = (val_probs.argmax(axis=1) == val_y).astype(float)
    q_inputs = np.zeros((len(val_conf), n_qubits), dtype=np.float64)
    q_inputs[:, 0] = np.clip(val_conf, 0, 1) * np.pi
    q_inputs[:, 1] = np.clip(1-val_conf, 0, 1) * np.pi
    q_inputs[:, 2] = np.clip(val_probs[:, 0], 0, 1) * np.pi
    q_inputs[:, 3] = np.clip(val_probs.std(axis=1), 0, 1) * np.pi

    def objective(flat):
        theta = flat.reshape(2, n_qubits)
        # Quantum expectation contributes a learnable calibration score.
        q_scores = np.array([np.mean(calibration_circuit(x, theta)) for x in q_inputs])
        q_scores = 1 / (1 + np.exp(-q_scores))
        threshold = 1/(1+np.exp(-flat[-1]))
        calibrated = np.clip(0.75*val_conf + 0.25*q_scores, 0, 1)
        pred_ok = calibrated >= threshold
        # Maximize correct acceptance while penalizing rejected correct predictions.
        return np.mean((pred_ok.astype(float)-val_correct)**2)
    init = np.zeros(2*n_qubits + 1)
    opt = minimize(objective, init, method="COBYLA", options={"maxiter": 35})
    quantum_threshold = float(1/(1+np.exp(-opt.x[-1])))
    quantum_weights = opt.x[:-1].reshape(2, n_qubits)
    QUANTUM_BACKEND = "PennyLane default.qubit (4 qubits, 2 variational layers)"
    print("Quantum calibration complete. Threshold:", quantum_threshold)
except Exception as e:
    print("Quantum backend unavailable or calibration failed:", repr(e))
    val_conf = val_probs.max(axis=1)
    # Classical validation-only threshold selection; no test data used.
    correct = (val_probs.argmax(1) == val_y).astype(int)
    candidates = np.linspace(0.35, 0.95, 61)
    scores = [np.mean(((val_conf >= t).astype(int) - correct)**2) for t in candidates]
    quantum_threshold = float(candidates[int(np.argmin(scores))])
    QUANTUM_BACKEND = "classical validation calibration fallback (not quantum)"
print("Calibration backend:", QUANTUM_BACKEND)
print("Retained suspicious/attack confidence threshold:", quantum_threshold)


In [ ]:
# Three-level confidence-aware decision policy.
def calibrated_decisions(probabilities, threshold=quantum_threshold):
    confidence = probabilities.max(axis=1)
    pred = probabilities.argmax(axis=1)
    # Below threshold: suspicious; above threshold: high-confidence predicted class.
    state = np.where(confidence >= threshold, "high-confidence", "suspicious")
    state[(pred == 0) & (confidence >= threshold)] = "benign"
    return pred, confidence, state

cal_pred, cal_conf, cal_state = calibrated_decisions(test_probs)
print(pd.Series(cal_state).value_counts())


## Stage 5 — Quantum-resistant key establishment and authenticated encryption

In [ ]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import x25519, ed25519
from cryptography.hazmat.primitives.serialization import Encoding, PublicFormat

def derive_aes_key(shared_secret, context=b"BQ-KFE-session-v1"):
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None, info=context).derive(shared_secret)

def aes_gcm_encrypt(key, plaintext: bytes, aad: bytes):
    nonce = os.urandom(12)
    ciphertext = AESGCM(key).encrypt(nonce, plaintext, aad)
    return nonce, ciphertext

def aes_gcm_decrypt(key, nonce, ciphertext, aad):
    return AESGCM(key).decrypt(nonce, ciphertext, aad)

# Probe liboqs algorithm support without falsely claiming PQ operation.
OQS_AVAILABLE = False
OQS_KEM_NAME = None
OQS_SIG_NAME = None
try:
    import oqs
    kem_names = oqs.get_enabled_kem_mechanisms()
    sig_names = oqs.get_enabled_sig_mechanisms()
    OQS_KEM_NAME = next((x for x in ["ML-KEM-768", "Kyber768"] if x in kem_names), None)
    OQS_SIG_NAME = next((x for x in ["ML-DSA-65", "Dilithium3"] if x in sig_names), None)
    OQS_AVAILABLE = bool(OQS_KEM_NAME and OQS_SIG_NAME)
except Exception as e:
    print("liboqs not available:", repr(e))

print("OQS available:", OQS_AVAILABLE, "| KEM:", OQS_KEM_NAME, "| Signature:", OQS_SIG_NAME)


In [ ]:
def establish_session_and_sign(payload: bytes, aad: bytes):
    if OQS_AVAILABLE:
        # NIST-standardized post-quantum KEM and signature, when enabled by liboqs.
        with oqs.KeyEncapsulation(OQS_KEM_NAME) as recipient:
            recipient_public = recipient.generate_keypair()
            ciphertext, shared_sender = oqs.KeyEncapsulation(OQS_KEM_NAME).encap_secret(recipient_public)
            shared_receiver = recipient.decap_secret(ciphertext)
        assert shared_sender == shared_receiver
        session_key = derive_aes_key(shared_sender)
        with oqs.Signature(OQS_SIG_NAME) as signer:
            sig_public = signer.generate_keypair()
            signature = signer.sign(hashlib.sha256(aad + payload).digest())
            valid = oqs.Signature(OQS_SIG_NAME).verify(
                hashlib.sha256(aad + payload).digest(), signature, sig_public)
        crypto_mode = f"POST-QUANTUM: {OQS_KEM_NAME} + {OQS_SIG_NAME}"
        public_credential = hashlib.sha256(recipient_public + sig_public).hexdigest()
    else:
        # Functional classical fallback only. This does NOT provide post-quantum security.
        recipient_private = x25519.X25519PrivateKey.generate()
        sender_private = x25519.X25519PrivateKey.generate()
        shared_sender = sender_private.exchange(recipient_private.public_key())
        shared_receiver = recipient_private.exchange(sender_private.public_key())
        assert shared_sender == shared_receiver
        session_key = derive_aes_key(shared_sender)
        signing_key = ed25519.Ed25519PrivateKey.generate()
        signature = signing_key.sign(hashlib.sha256(aad + payload).digest())
        signing_key.public_key().verify(signature, hashlib.sha256(aad + payload).digest())
        valid = True
        crypto_mode = "CLASSICAL FALLBACK: X25519 + Ed25519 (NOT post-quantum)"
        public_credential = hashlib.sha256(
            recipient_private.public_key().public_bytes(Encoding.Raw, PublicFormat.Raw)
        ).hexdigest()

    nonce, ciphertext = aes_gcm_encrypt(session_key, payload, aad)
    recovered = aes_gcm_decrypt(session_key, nonce, ciphertext, aad)
    assert recovered == payload and valid
    return {"nonce": nonce, "ciphertext": ciphertext, "signature": signature,
            "public_credential": public_credential, "crypto_mode": crypto_mode,
            "key_id": hashlib.sha256(session_key).hexdigest()[:16],
            "session_key": session_key}

demo_payload = b"simulated-healthcare-payload: heart-rate=72"
demo_aad = b"device=patient-monitor-001;role=patient-monitor"
crypto_demo = establish_session_and_sign(demo_payload, demo_aad)
print({k:v for k,v in crypto_demo.items() if k not in {"session_key", "ciphertext", "nonce", "signature"}})
print("Encrypted bytes:", len(crypto_demo["ciphertext"]))


## Stage 6 — Blockchain-verified, key-escrow-free access control

In [ ]:
# Executable local permissioned-ledger prototype:
# only hashes, key IDs, identities/fingerprints, policy events and metadata are recorded.
# Clinical plaintext and private/session keys remain off-chain.
class HashChainedLedger:
    def __init__(self):
        self.blocks = []
        self.append("GENESIS", {"network": "local-permissioned-prototype"})
    def append(self, event_type, payload):
        previous = self.blocks[-1]["hash"] if self.blocks else "0"*64
        body = {"index": len(self.blocks), "timestamp": time.time(),
                "event_type": event_type, "payload": payload, "previous_hash": previous}
        digest = hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()
        block = {**body, "hash": digest}
        self.blocks.append(block)
        return digest
    def verify(self):
        for i, block in enumerate(self.blocks):
            body = {k:block[k] for k in ["index","timestamp","event_type","payload","previous_hash"]}
            expected = hashlib.sha256(json.dumps(body, sort_keys=True).encode()).hexdigest()
            if expected != block["hash"]: return False
            if i and block["previous_hash"] != self.blocks[i-1]["hash"]: return False
        return True

ledger = HashChainedLedger()
device_registry = {}
revoked_devices = set()

def enroll_device(device_id, role, public_fingerprint):
    device_registry[device_id] = {"role": role, "fingerprint": public_fingerprint}
    return ledger.append("DEVICE_ENROLLMENT", {
        "device_id": device_id, "role": role, "public_key_fingerprint": public_fingerprint})

def authorize_access(device_id, resource, requested_action, risk_state):
    if device_id not in device_registry or device_id in revoked_devices:
        allowed, reason = False, "unknown-or-revoked-device"
    elif risk_state == "high-confidence" and requested_action != "quarantine":
        allowed, reason = False, "high-confidence-threat"
    elif risk_state == "suspicious" and requested_action not in {"read_status", "send_telemetry"}:
        allowed, reason = False, "restricted-suspicious-device"
    else:
        allowed, reason = True, "least-privilege-policy-permitted"
    ledger.append("ACCESS_DECISION", {"device_id": device_id, "resource": resource,
                  "action": requested_action, "risk_state": risk_state,
                  "allowed": allowed, "reason": reason})
    return allowed, reason

def revoke_device(device_id, reason):
    revoked_devices.add(device_id)
    ledger.append("CREDENTIAL_REVOCATION", {"device_id": device_id, "reason": reason})

enroll_device("patient-monitor-001", "patient-monitor", crypto_demo["public_credential"])
print(authorize_access("patient-monitor-001", "encrypted-vitals", "send_telemetry", "benign"))
print("Ledger valid:", ledger.verify(), "| blocks:", len(ledger.blocks))


In [ ]:
# Demonstrate off-chain encrypted payload storage and on-chain hash-only reference.
encrypted_blob = crypto_demo["nonce"] + crypto_demo["ciphertext"]
payload_hash = hashlib.sha256(encrypted_blob).hexdigest()
ledger.append("ENCRYPTED_PAYLOAD_REGISTERED", {
    "device_id": "patient-monitor-001", "payload_hash": payload_hash,
    "key_id": crypto_demo["key_id"], "storage_uri": "offchain://secure-storage/object-001"})
print("On-chain payload hash:", payload_hash)
print("Ledger integrity:", ledger.verify())


## Stage 7 — Integrated security orchestration and mitigation

In [ ]:
def orchestrate_security(device_id, predicted_class, confidence, crypto_verified=True):
    predicted_name = class_names[int(predicted_class)]
    # Risk combines model confidence, malicious-class prediction and crypto verification.
    predicted_benign = predicted_name.lower() in {"benign", "normal"}
    risk = 0.0 if predicted_benign else float(confidence)
    if not crypto_verified: risk = max(risk, 0.95)
    if risk >= 0.85:
        action = "quarantine"
        revoke_device(device_id, "high-risk security orchestration")
        decision = "QUARANTINE"
    elif risk >= 0.50:
        action = "send_telemetry"
        decision = "RESTRICT"
    else:
        action = "send_telemetry"
        decision = "PERMIT"
    allowed, reason = authorize_access(device_id, "healthcare-data", action,
                                       "high-confidence" if risk >= 0.85 else
                                       "suspicious" if risk >= 0.50 else "benign")
    ledger.append("MITIGATION_ACTION", {"device_id": device_id, "risk_score": risk,
                  "decision": decision, "policy_result": reason})
    return {"device_id": device_id, "predicted_attack": predicted_name,
            "confidence": float(confidence), "risk_score": risk,
            "decision": decision, "authorization": allowed, "reason": reason}

# Apply orchestration to a bounded test subset for inspectable audit records.
orchestration_results = []
for i in range(min(100, len(cal_pred))):
    orchestration_results.append(orchestrate_security(
        f"test-device-{i}", cal_pred[i], cal_conf[i], crypto_verified=True))
display(pd.DataFrame(orchestration_results).head(10))
print("Ledger integrity after orchestration:", ledger.verify())


## Stage 8 — Evaluation, baselines, ablation, and efficiency

In [ ]:
# Multi-class metrics for the proposed model.
y_pred = test_probs.argmax(axis=1)
metrics = {
    "Accuracy": accuracy_score(test_y, y_pred),
    "Precision_macro": precision_score(test_y, y_pred, average="macro", zero_division=0),
    "Recall_macro": recall_score(test_y, y_pred, average="macro", zero_division=0),
    "Macro_F1": f1_score(test_y, y_pred, average="macro", zero_division=0),
}
try:
    metrics["AUROC_ovr_macro"] = roc_auc_score(test_y, test_probs, multi_class="ovr", average="macro")
except Exception:
    metrics["AUROC_ovr_macro"] = np.nan
print(pd.Series(metrics))
print(classification_report(test_y, y_pred, labels=np.arange(len(class_names)),
                            target_names=class_names, zero_division=0))


In [ ]:
# Confusion matrix
cm = confusion_matrix(test_y, y_pred, labels=np.arange(len(class_names)))
fig, ax = plt.subplots(figsize=(max(7, len(class_names)*0.8), max(5, len(class_names)*0.65)))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label"); ax.set_title("Q-DCTD Test Confusion Matrix")
fig.colorbar(im, ax=ax); plt.tight_layout(); plt.show()


In [ ]:
# Binary benign-vs-malicious metrics and ROC. The benign class is identified by label text.
benign_ids = [i for i,n in enumerate(class_names) if n.lower() in {"benign","normal"}]
if benign_ids:
    benign_id = benign_ids[0]
    y_bin = (test_y != benign_id).astype(int)
    p_attack = 1.0 - test_probs[:, benign_id]
    fpr, tpr, _ = roc_curve(y_bin, p_attack)
    plt.figure(figsize=(6,5)); plt.plot(fpr, tpr, label=f"Q-DCTD (AUC={roc_auc_score(y_bin,p_attack):.4f})")
    plt.plot([0,1],[0,1],"--",label="Chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("Binary Attack Detection ROC"); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()
    print("Binary FPR:", float(np.mean((p_attack >= 0.5) & (y_bin == 0))))
else:
    print("Benign class not found; binary ROC omitted.")


In [ ]:
# Baseline implementations: single-channel TCN variants and conventional classifiers.
# Flattened-window baselines are trained only on training windows and evaluated on held-out test windows.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def flatten_pair(a, b=None):
    if b is None: return a.reshape(len(a), -1)
    return np.concatenate([a.reshape(len(a), -1), b.reshape(len(b), -1)], axis=1)

baseline_rows = []
baseline_specs = [
    ("Logistic Regression", LogisticRegression(max_iter=500, class_weight="balanced")),
    ("Random Forest", RandomForestClassifier(n_estimators=250, max_depth=None,
                                              class_weight="balanced_subsample",
                                              n_jobs=-1, random_state=SEED)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(max_iter=150, random_state=SEED))
]
for name, clf in baseline_specs:
    xb_tr, xb_te = flatten_pair(Xn_train, Xd_train), flatten_pair(Xn_test, Xd_test)
    start = time.perf_counter()
    clf.fit(xb_tr, y_train)
    pred = clf.predict(xb_te)
    elapsed = time.perf_counter()-start
    baseline_rows.append({"Model": name, "Accuracy": accuracy_score(y_test,pred),
                          "Precision_macro": precision_score(y_test,pred,average="macro",zero_division=0),
                          "Recall_macro": recall_score(y_test,pred,average="macro",zero_division=0),
                          "Macro_F1": f1_score(y_test,pred,average="macro",zero_division=0),
                          "Fit+test_seconds": elapsed})

# Single-channel logistic baselines provide direct channel ablations.
for name, xtr, xte in [
    ("Network-only Logistic", flatten_pair(Xn_train), flatten_pair(Xn_test)),
    ("Device/proxy-only Logistic", flatten_pair(Xd_train), flatten_pair(Xd_test))
]:
    clf = LogisticRegression(max_iter=500, class_weight="balanced")
    start = time.perf_counter(); clf.fit(xtr, y_train); pred = clf.predict(xte)
    baseline_rows.append({"Model": name, "Accuracy": accuracy_score(y_test,pred),
                          "Precision_macro": precision_score(y_test,pred,average="macro",zero_division=0),
                          "Recall_macro": recall_score(y_test,pred,average="macro",zero_division=0),
                          "Macro_F1": f1_score(y_test,pred,average="macro",zero_division=0),
                          "Fit+test_seconds": time.perf_counter()-start})

baseline_table = pd.DataFrame(baseline_rows)
display(baseline_table)


In [ ]:
# Ablation: train separate Q-DCTD variants with one channel zeroed at inference.
# This isolates reliance on each channel without retraining a second architecture.
model.eval()
with torch.no_grad():
    tn = torch.tensor(Xn_test, dtype=torch.float32, device=DEVICE)
    td = torch.tensor(Xd_test, dtype=torch.float32, device=DEVICE)
    logits_full = model(tn, td)[0].cpu().numpy()
    logits_network_removed = model(tn, torch.zeros_like(td))[0].cpu().numpy()
    logits_device_removed = model(torch.zeros_like(tn), td)[0].cpu().numpy()

def score_logits(logits):
    pred = logits.argmax(1)
    return {"Accuracy": accuracy_score(y_test,pred),
            "Macro_F1": f1_score(y_test,pred,average="macro",zero_division=0)}
ablation = pd.DataFrame([
    {"Variant":"Full Q-DCTD", **score_logits(logits_full)},
    {"Variant":"Device channel removed", **score_logits(logits_device_removed)},
    {"Variant":"Network channel removed", **score_logits(logits_network_removed)}
])
display(ablation)


In [ ]:
# Cryptographic benchmark: key establishment + HKDF + AES-GCM + signature path.
N_CRYPTO = 100
crypto_times = []
for i in range(N_CRYPTO):
    payload = os.urandom(1024)
    aad = f"device=benchmark-{i};role=gateway".encode()
    t0 = time.perf_counter()
    result = establish_session_and_sign(payload, aad)
    crypto_times.append(time.perf_counter()-t0)
print("Crypto mode:", crypto_demo["crypto_mode"])
print("Mean end-to-end crypto operation (ms):", np.mean(crypto_times)*1000)
print("P95 (ms):", np.percentile(crypto_times,95)*1000)


In [ ]:
# Ledger performance benchmark.
N_LEDGER = 1000
t0 = time.perf_counter()
for i in range(N_LEDGER):
    ledger.append("BENCHMARK_EVENT", {"event_id": i, "payload_hash": hashlib.sha256(str(i).encode()).hexdigest()})
ledger_write_s = time.perf_counter()-t0
t0 = time.perf_counter(); ledger_valid = ledger.verify(); ledger_verify_s = time.perf_counter()-t0
ledger_json_bytes = len(json.dumps(ledger.blocks).encode())
print({"events": N_LEDGER, "write_events_per_second": N_LEDGER/ledger_write_s,
       "verification_seconds": ledger_verify_s, "serialized_ledger_bytes": ledger_json_bytes,
       "integrity_valid": ledger_valid})


In [ ]:
# Quantum resource profile (when PennyLane circuit ran) and complete experiment summary.
resource_profile = {
    "quantum_backend": QUANTUM_BACKEND,
    "qubit_count": 4 if "PennyLane" in QUANTUM_BACKEND else 0,
    "variational_layers": 2 if "PennyLane" in QUANTUM_BACKEND else 0,
    "circuit_execution_time": "measured during calibration; see cell output" if "PennyLane" in QUANTUM_BACKEND else "not executed",
    "device_channel": "measured telemetry" if telemetry_is_genuine else "network-derived proxy; not measured telemetry",
    "cryptography": crypto_demo["crypto_mode"],
    "ledger": "local hash-chained prototype; not Hyperledger Fabric"
}
summary = {
    "dataset_rows": int(len(data)),
    "dataset_files": [p.name for p in csv_files],
    "window_size": WINDOW, "stride": STRIDE,
    "train_windows": int(len(y_train)), "validation_windows": int(len(y_val)),
    "test_windows": int(len(y_test)), "classes": class_names,
    "metrics": {k: float(v) for k,v in metrics.items()},
    "resources": resource_profile
}
print(json.dumps(summary, indent=2))


In [ ]:
# Save trained weights and experiment metadata locally.
OUTPUT_DIR = Path("BQ_KFE_results")
OUTPUT_DIR.mkdir(exist_ok=True)
torch.save({"model_state_dict": model.state_dict(),
            "network_features": network_cols, "device_features": device_features,
            "class_names": class_names, "window": WINDOW, "stride": STRIDE},
           OUTPUT_DIR / "qdctd_model.pt")
with open(OUTPUT_DIR / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
baseline_table.to_csv(OUTPUT_DIR / "baseline_comparison.csv", index=False)
ablation.to_csv(OUTPUT_DIR / "ablation_results.csv", index=False)
pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Saved outputs to:", OUTPUT_DIR.resolve())


## Optional extension — Hyperledger Fabric deployment mapping

The local ledger cells above are an executable prototype for event schema, hash chaining, and policy logic. To turn Stage 6 into a real Fabric deployment:

1. Create a Fabric test network with separate organizations for medical devices/gateways and hospital infrastructure.
2. Enroll identities through Fabric CA; map device roles to certificate attributes.
3. Implement chaincode transactions corresponding to `DEVICE_ENROLLMENT`, `ACCESS_DECISION`, `KEY_VERSION_UPDATE`, `CREDENTIAL_REVOCATION`, `PAYLOAD_REGISTERED`, and `MITIGATION_ACTION`.
4. Store only ciphertext references, hashes, key identifiers, fingerprints, and audit metadata on-chain.
5. Keep ML-KEM private keys and derived session secrets on endpoint devices or protected hardware.
6. Replace the `HashChainedLedger.append()` calls with Fabric Gateway SDK submissions and verify commit status before treating an action as finalized.

## Experimental validity checklist

- Use full dataset for final results; `ROW_LIMIT` is only for development smoke tests.
- Keep all windows from a device/time partition within a single split.
- For stronger device-held-out evaluation, reserve complete device IDs for test rather than only chronological splits within each device.
- Report the actual `crypto_mode`, quantum backend, dataset telemetry availability, and ledger implementation alongside every result.
- Run ToN-IoT cross-dataset validation only after harmonizing feature semantics; do not directly feed mismatched raw columns.
- For a strict multi-class benign/malicious task, verify the dataset's raw attack-label taxonomy and adjust `clean_label()` if the release uses different label spellings.
